# 09 · Comparar reranking con cross-encoders sobre nomic-embed-text en Qdrant

Medimos si reordenar los 20 candidatos de la búsqueda con un cross-encoder mejora lo que ya recupera k = 4. Se comparan tres configuraciones: sin reranking (referencia, decisión vigente de `docs/FICHA_04_NUMERO_K.md`), `cross-encoder/ms-marco-MiniLM-L-6-v2` y `BAAI/bge-reranker-base`. Se fijan el modelo de embeddings (`nomic-embed-text`, Ollama), la base (Qdrant) y la colección `pydocs_nomic_est1500_m200_sinruta`. No se indexa nada: el notebook solo consulta. Se evalúa con el golden set de `docs/GOLDEN_SET_PYTHON_DOCS.md` y las evidencias de `docs/golden_set_evidencias.json`.

Necesita Qdrant en `localhost:6333` (`docker compose up -d`), Ollama con `nomic-embed-text` (`ollama pull nomic-embed-text`) y la colección creada con `notebooks/07_comparar_troceo_estructura.ipynb`. La primera ejecución descarga los dos rerankers de Hugging Face.

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

`CONFIGURACIONES` es el único sitio donde se declaran las configuraciones y sus rerankers. El nombre de la colección sale de `MODELO`, `MAXIMO`, `MINIMO` y `CON_RUTA`, con el formato de `nombre_coleccion` del notebook 07. `MEJORA_MINIMA` (preguntas) y `LATENCIA_MAXIMA_MS` son los umbrales del criterio de decisión.

In [ ]:
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
from qdrant_client import QdrantClient
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, elegir_reranker, fuentes_ausentes, leer_evidencias, leer_golden_set,
    preparar_query, rango_primer_acierto, rango_primer_acierto_evidencia, reordenar_por_score,
    trocear_por_estructura,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CSV = RAIZ / "data" / "resultados" / "reranking.csv"
NOTEBOOK_INDEXADO = "notebooks/07_comparar_troceo_estructura.ipynb"
MODELO = "nomic"
MAXIMO = 1500
MINIMO = 200
CON_RUTA = False
K = 4
CANDIDATOS = 20
REFERENCIA = "sin reranking"
CONFIGURACIONES = [
    {"nombre": REFERENCIA, "reranker": None},
    {"nombre": "MiniLM-L6", "reranker": "cross-encoder/ms-marco-MiniLM-L-6-v2"},
    {"nombre": "bge-reranker-base", "reranker": "BAAI/bge-reranker-base"},
]
MEJORA_MINIMA = 2
LATENCIA_MAXIMA_MS = 500


def nombre_coleccion(modelo: str, maximo: int, minimo: int, con_ruta: bool) -> str:
    sufijo = "ruta" if con_ruta else "sinruta"
    return f"pydocs_{modelo}_est{maximo}_m{minimo}_{sufijo}"


COLECCION = nombre_coleccion(MODELO, MAXIMO, MINIMO, CON_RUTA)
print("Colección:", COLECCION)

## 3. Conexión con Qdrant y Ollama

Se comprueba antes de calcular nada, para fallar pronto si falta un servicio. Si Qdrant u Ollama no responden, o Ollama no tiene el modelo, la celda para con un mensaje que dice qué hacer.

In [ ]:
qdrant = QdrantClient("localhost", port=6333)
try:
    qdrant.get_collections()
except Exception as error:
    raise RuntimeError(
        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d` y vuelve a ejecutar."
    ) from error

modelo_ollama = MODELOS[MODELO]["nombre"]
try:
    ollama.show(modelo_ollama)
except ConnectionError as error:
    raise RuntimeError("Ollama no responde. Arráncalo (`ollama serve` o la app) y vuelve a ejecutar.") from error
except ollama.ResponseError as error:
    raise RuntimeError(
        f"Ollama no tiene el modelo {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}` y vuelve a ejecutar."
    ) from error

print("Qdrant:", [col.name for col in qdrant.get_collections().collections])
print("Ollama:", modelo_ollama, "disponible")

## 4. Comprobación de la colección

El notebook no crea, borra ni modifica colecciones. Para antes de evaluar si la colección no existe o si su número de puntos no coincide con el de chunks del troceo por estructura.

In [ ]:
if not qdrant.collection_exists(COLECCION):
    raise RuntimeError(f"La colección {COLECCION} no existe. Créala con `{NOTEBOOK_INDEXADO}`.")

puntos = qdrant.count(COLECCION, exact=True).count
n_chunks = len(trocear_por_estructura(CARPETA_CORPUS, MAXIMO, MINIMO))
if puntos != n_chunks:
    raise RuntimeError(
        f"{COLECCION} tiene {puntos} puntos y el troceo {MAXIMO}/{MINIMO} da {n_chunks} chunks. "
        f"No la toco: recréala con `{NOTEBOOK_INDEXADO}`."
    )
print(f"{COLECCION}: {puntos} puntos, igual que los chunks del troceo")

## 5. Preguntas y evidencias del golden set

Se leen de `docs/GOLDEN_SET_PYTHON_DOCS.md` y `docs/golden_set_evidencias.json`; nada está copiado en el código. Se avisa, con el ID y la ruta, de las fuentes que no existen en el corpus.

In [ ]:
preguntas = leer_golden_set(RUTA_GOLDEN)
ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)
for id_, ruta in ausentes:
    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")
print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())

evidencias = leer_evidencias(RUTA_EVIDENCIAS)

## 6. Carga de los rerankers

Se construye cada `CrossEncoder` midiendo el tiempo de carga (en la primera ejecución incluye la descarga de Hugging Face). Después se hace una llamada de calentamiento a `predict`, fuera de la medición de latencia. Si un modelo no se puede descargar o cargar, la celda para con su nombre.

In [ ]:
rerankers = {}
carga_s = {}
for config in CONFIGURACIONES:
    nombre_modelo = config["reranker"]
    carga_s[config["nombre"]] = 0.0
    if nombre_modelo is None:
        continue
    inicio = time.perf_counter()
    try:
        rerankers[config["nombre"]] = CrossEncoder(nombre_modelo)
    except Exception as error:
        raise RuntimeError(f"No se pudo cargar el reranker {nombre_modelo}: {error}") from error
    carga_s[config["nombre"]] = time.perf_counter() - inicio
    rerankers[config["nombre"]].predict([("calentamiento", "calentamiento")])
    print(f"{nombre_modelo}: cargado en {carga_s[config['nombre']]:.1f} s, dispositivo {rerankers[config['nombre']].device}")

## 7. Evaluación por configuración

Para cada configuración y cada pregunta (A-D y E): la referencia hace una búsqueda propia con `limit=4`; cada reranker hace una con `limit=20`, puntúa los pares (query sin prefijo de nomic, `texto` del chunk) con `CrossEncoder.predict` y se queda con los 4 de mayor score (`reordenar_por_score`). Si la búsqueda devuelve menos de 20, se reordenan los que haya. Se guardan el rango por fuente y por evidencia (solo A-D), la latencia de búsqueda (embedding más Qdrant), la latencia de reranking y el score del primer resultado reordenado.

In [ ]:
def embeber(texto: str) -> list[float]:
    return ollama.embed(model=modelo_ollama, input=[texto]).embeddings[0]


resultados = []
for config in CONFIGURACIONES:
    nombre = config["nombre"]
    for p in tqdm(preguntas, desc=nombre):
        inicio = time.perf_counter()
        vector = embeber(preparar_query(MODELO, p["query"]))
        limite = CANDIDATOS if config["reranker"] else K
        res = qdrant.query_points(COLECCION, query=vector, limit=limite, with_payload=True).points
        busqueda_ms = (time.perf_counter() - inicio) * 1000
        candidatos = [{"fuente": r.payload["fuente"], "texto": r.payload["texto"]} for r in res]

        rerank_ms, score_primero = 0.0, None
        if config["reranker"]:
            inicio = time.perf_counter()
            scores = [float(s) for s in rerankers[nombre].predict([(p["query"], c["texto"]) for c in candidatos])]
            recuperados = reordenar_por_score(candidatos, scores, K)
            rerank_ms = (time.perf_counter() - inicio) * 1000
            score_primero = max(scores)
        else:
            recuperados = candidatos

        fila = {"configuracion": nombre, "id": p["id"], "categoria": p["categoria"],
                "busqueda_ms": busqueda_ms, "rerank_ms": rerank_ms, "score_primero": score_primero,
                "rango": None, "rango_evidencia": None}
        if p["fuentes"]:
            fila["rango"] = rango_primer_acierto([r["fuente"] for r in recuperados], p["fuentes"])
            fila["rango_evidencia"] = rango_primer_acierto_evidencia(recuperados, evidencias[p["id"]])
        resultados.append(fila)

df = pd.DataFrame(resultados)
df_abcd = df[df["categoria"] != "E"]
print(len(df), "evaluaciones,", len(df_abcd), "con fuentes (A-D)")

## 8. Métricas por configuración (preguntas A-D)

hit@1, hit@4 y MRR@4 con los dos criterios: por fuente y por evidencia (más estricto, prefijo `ev_`). También los aciertos contados (de 32), que son los que usa el criterio.

In [ ]:
def metricas(grupo: pd.DataFrame) -> dict:
    """Métricas por fuente y por evidencia (prefijo `ev_`) con k = K."""
    filas = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        m = calcular_metricas(rangos, K)
        filas.update({prefijo + "hit@1": m["hit@1"], prefijo + f"hit@{K}": m[f"hit@{K}"], prefijo + "MRR": m["mrr"]})
    return filas


por_config = pd.DataFrame({c["nombre"]: metricas(df_abcd[df_abcd["configuracion"] == c["nombre"]])
                           for c in CONFIGURACIONES}).T
n_abcd = df_abcd["id"].nunique()
por_config["aciertos hit@4"] = (por_config[f"hit@{K}"] * n_abcd).round().astype(int)
por_config["aciertos ev_hit@4"] = (por_config[f"ev_hit@{K}"] * n_abcd).round().astype(int)
por_config.round(3)

## 9. Desglose por categoría

hit@4 y ev_hit@4 por configuración y categoría A-D.

In [ ]:
filas_cat = []
for config in CONFIGURACIONES:
    sub = df_abcd[df_abcd["configuracion"] == config["nombre"]]
    for categoria, grupo in sub.groupby("categoria"):
        m = metricas(grupo)
        filas_cat.append({"configuración": config["nombre"], "categoría": categoria, "preguntas": len(grupo),
                          f"hit@{K}": m[f"hit@{K}"], f"ev_hit@{K}": m[f"ev_hit@{K}"]})
por_categoria = pd.DataFrame(filas_cat)
por_categoria.round(3)

## 10. Preguntas E frente a A-D: score del primer resultado

Las preguntas E no tienen respuesta en el corpus. Por reranker se muestra el score más alto tras puntuar los candidatos en las E, junto a la media y el mínimo del mismo score en A-D. La referencia no tiene score de reranking. Estos scores no van al CSV.

In [ ]:
filas_e = []
for config in CONFIGURACIONES:
    if config["reranker"] is None:
        continue
    sub = df[df["configuracion"] == config["nombre"]]
    abcd = sub[sub["categoria"] != "E"]["score_primero"]
    for _, r in sub[sub["categoria"] == "E"].iterrows():
        filas_e.append({"configuración": config["nombre"], "pregunta E": r["id"], "score primero": r["score_primero"],
                        "media A-D": abcd.mean(), "mínimo A-D": abcd.min()})
pd.DataFrame(filas_e).round(3)

## 11. Criterio de decisión

`elegir_reranker` trabaja con aciertos contados: un reranker cumple si mejora los aciertos ev_hit@4 de la referencia en al menos `MEJORA_MINIMA`, no baja los aciertos hit@4 por fuente y su latencia de reranking media es menor que `LATENCIA_MAXIMA_MS`. Gana el de más aciertos por evidencia y, si empatan, el de menor latencia. Si ninguno cumple, la decisión es no usar reranking.

In [ ]:
lat_rerank = df_abcd.groupby("configuracion")["rerank_ms"].mean()
ref = por_config.loc[REFERENCIA]
referencia = {"ev_aciertos": int(ref["aciertos ev_hit@4"]), "aciertos": int(ref["aciertos hit@4"])}
opciones = [
    {"nombre": c["nombre"], "ev_aciertos": int(por_config.loc[c["nombre"], "aciertos ev_hit@4"]),
     "aciertos": int(por_config.loc[c["nombre"], "aciertos hit@4"]), "latencia_ms": float(lat_rerank[c["nombre"]])}
    for c in CONFIGURACIONES if c["reranker"]
]
filas_criterio, ganador = elegir_reranker(referencia, opciones, MEJORA_MINIMA, LATENCIA_MAXIMA_MS)
print("Referencia:", referencia)
print("Decisión:", ganador if ganador else "sin reranking (ningún reranker cumple)")
pd.DataFrame(filas_criterio)

## 12. Tabla resumen y CSV

Una fila por configuración, con las mismas columnas para todas. Se guarda en `data/resultados/reranking.csv` (la carpeta se crea si no existe). En la referencia el modelo es «ninguno», los candidatos son 4 y la latencia de reranking y la carga valen 0. Si hubiera una celda vacía, la celda falla antes de escribir.

In [ ]:
filas = []
for config in CONFIGURACIONES:
    nombre = config["nombre"]
    sub = df_abcd[df_abcd["configuracion"] == nombre]
    m = por_config.loc[nombre]
    fila = {
        "configuración": nombre,
        "modelo de reranking": config["reranker"] or "ninguno",
        "candidatos": CANDIDATOS if config["reranker"] else K,
        "hit@1": m["hit@1"], "hit@4": m["hit@4"], "MRR": m["MRR"],
        "ev_hit@1": m["ev_hit@1"], "ev_hit@4": m["ev_hit@4"], "ev_MRR": m["ev_MRR"],
    }
    for categoria in sorted(df_abcd["categoria"].unique()):
        mc = metricas(sub[sub["categoria"] == categoria])
        fila[f"hit@4 {categoria}"] = mc["hit@4"]
        fila[f"ev_hit@4 {categoria}"] = mc["ev_hit@4"]
    fila["latencia búsqueda (ms)"] = sub["busqueda_ms"].mean()
    fila["latencia reranking (ms)"] = sub["rerank_ms"].mean()
    fila["latencia total (ms)"] = fila["latencia búsqueda (ms)"] + fila["latencia reranking (ms)"]
    fila["carga (s)"] = carga_s[nombre]
    filas.append(fila)
resumen = pd.DataFrame(filas)
assert not resumen.isna().any().any(), "El CSV tendría celdas vacías"
RUTA_CSV.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CSV, index=False)
print(f"Guardado en {RUTA_CSV.relative_to(RAIZ)}")
resumen.round(3)